In [77]:
import pandas as pd, os
from sklearn.preprocessing import minmax_scale

In [26]:
df = pd.read_csv("../src/data/files/F2_Sample_data_last6_campaigns.csv")
# df_mae_p = pd.read_csv("mae_producto_20rows.csv")

"""
Columns
= alternative
== same
1. Tipo subestrategia == DES_TIPO_SUBESTRATEGIA
2. Tipo grupo == DES_TIPO_GRUPO
3. Indicator padre == ES_PADRE
4. Indicator gratis = ES_GRATIS
5. Factor of repetition == FACTOR_REPETICION

"""

'\nColumns\n= alternative\n== same\n1. Tipo subestrategia == DES_TIPO_SUBESTRATEGIA\n2. Tipo grupo == DES_TIPO_GRUPO\n3. Indicator padre == ES_PADRE\n4. Indicator gratis = ES_GRATIS\n5. Factor of repetition == FACTOR_REPETICION\n\n'

In [27]:
unique_occurrences = df['ANIOCAMPANA'].value_counts()

print("Unique occurrences:")
print(unique_occurrences)

Unique occurrences:
ANIOCAMPANA
202412    2125
202413    1975
202414    1264
202415     960
202416     633
202417     187
202418       2
Name: count, dtype: int64


In [28]:
# Divide the DataFrame based on unique values in the 'Category' column
dfs = {category: df_subset for category, df_subset in df.groupby('ANIOCAMPANA')}
i=1
# Dynamically create variable names for each sub_df
for category, sub_df in dfs.items():
    # Create a unique name for each sub_df (based on category)
    variable_name = f"df_{i}"
    
    # Assign the sub_df to the unique variable name
    globals()[variable_name] = sub_df
    
    # Optionally, print the name and content to verify
    print(f"Created variable: {variable_name}")
    i+=1

Created variable: df_1
Created variable: df_2
Created variable: df_3
Created variable: df_4
Created variable: df_5
Created variable: df_6
Created variable: df_7


In [37]:
# Assuming `dfs` is a dictionary containing the 14 DataFrames
# Define the function for recency calculation
def cal_recency(ref_date, last_purchase_date):
    last_purchase_date = pd.Timestamp(last_purchase_date)
    days_difference = (ref_date - last_purchase_date).days
    recency = 1 / (days_difference + 1)  
    return recency

# Reference date for recency calculation
reference_date = pd.Timestamp('2024-12-29')  # You can replace this with the current date if needed

all_dfs = []
# Loop through each DataFrame, apply transformations, and save them
for i, (category, df) in enumerate(dfs.items(), 1):
    # Step 1: Filter the DataFrame
    # Get all ID_OFERTAs that have CODCUC as 'XXXXXXXXX'
    invalid_ids = df[df['CODCUC'] == 'XXXXXXXXX']['ID_OFERTA'].unique()
    
    # Filter out all rows with these ID_OFERTAs
    df_filtered = df[~df['ID_OFERTA'].isin(invalid_ids)]

    # Step 2: Create Composite_key column
    df_filtered["Composite_key"] = df_filtered[[
        "DES_TIPO_SUBESTRATEGIA", "DES_TIPO_GRUPO", "CODCUC", "ES_PADRE", 
        "ES_GRATIS", "FACTOR_REPETICION"
    ]].astype(str).agg('|'.join, axis=1)

    # Step 3: Drop the "COMPOSITE_PRIMARY_KEY" column
    df_filtered = df_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1, errors='ignore')

    # Step 4: Calculate recency
    recency_values = [cal_recency(reference_date, date) for date in df_filtered['FECHAPROCESO']]
    df_filtered["recency"] = recency_values

    # Step 5: Concatenate the grouped data
    concatenateds = df_filtered.groupby("ID_OFERTA")[["Composite_key", "CODEBELISTA", "recency"]].agg(
        {
            'CODEBELISTA': lambda x: x.dropna().tolist(),
            'Composite_key': lambda x: '|'.join(x),
            'recency': 'mean'  # Aggregating recency by mean
        }
    ).reset_index()

    # Step 6: Expand the CODEBELISTA column into separate rows
    expanded_df = concatenateds.explode('CODEBELISTA')

    # Step 7: Group by CODEBELISTA and Composite_key, aggregate ID_OFERTA into a list
    grouped_df = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).agg({
        'ID_OFERTA': lambda x: x.tolist(),
        'recency': 'mean'
    }).reset_index()

    # Step 8: Add count column
    grouped_df['count'] = expanded_df.groupby(['CODEBELISTA', 'Composite_key']).size().values

    # Step 9: Normalize the count and recency columns
    grouped_df['normalized_count'] = minmax_scale(grouped_df['count'])
    grouped_df['normalized_recency'] = minmax_scale(grouped_df['recency'])

    # Step 10: Calculate the score
    def score_calculation(recency, frequency, weight1=0.3, weight2=0.7):
        return weight1 * recency + frequency * weight2

    grouped_df['score'] = grouped_df.apply(lambda row: score_calculation(row['normalized_recency'], row['normalized_count']), axis=1)

    # Step 11: Sort by score in descending order
    sorted_df = grouped_df.sort_values('score', ascending=False)
    
    all_dfs.append(sorted_df)
    # Step 12: Save the result to a CSV file
    file_name = f"{category}.csv"
    sorted_df.to_csv(file_name, index=False)

    print(f"Saved {file_name}")

final_df = pd.concat(all_dfs, ignore_index=True)

Saved 202412.csv
Saved 202413.csv
Saved 202414.csv
Saved 202415.csv
Saved 202416.csv
Saved 202417.csv
Saved 202418.csv


/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_1271/2655493554.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Composite_key"] = df_filtered[[
/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_1271/2655493554.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered["Composite_key"] = df_filtered[[
/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_1271/2655493554.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Dat

In [52]:
single_campaign_df_202412 = pd.read_csv("202412.csv")
single_campaign_df_202413  = pd.read_csv("202413.csv")

In [53]:
single_campaign_df_202412_filtered = single_campaign_df_202412.groupby("CODEBELISTA").agg({
        "Composite_key": list,       # Keep Composite_key as a list
        "ID_OFERTA": lambda x: x.iloc[0] if all(x == x.iloc[0]) else list(x),  # Single value if all identical
        "recency": list,            # Keep recency as a list
        "count": list,              # Keep count as a list
        "normalized_count": list,   # Keep normalized_count as a list
        "normalized_recency": list, # Keep normalized_recency as a list
        "score": list               # Keep score as a list
    }).reset_index()

single_campaign_df_202413_filtered = single_campaign_df_202413.groupby("CODEBELISTA").agg({
        "Composite_key": list,       # Keep Composite_key as a list
        "ID_OFERTA": lambda x: x.iloc[0] if all(x == x.iloc[0]) else list(x),  # Single value if all identical
        "recency": list,            # Keep recency as a list
        "count": list,              # Keep count as a list
        "normalized_count": list,   # Keep normalized_count as a list
        "normalized_recency": list, # Keep normalized_recency as a list
        "score": list               # Keep score as a list
    }).reset_index()

single_campaign_df_202412_filtered.to_csv("Consultora_based_offers_202412.csv",index=False)
single_campaign_df_202413_filtered.to_csv("Consultora_based_offers_202413.csv",index=False)

In [50]:
single_campaign_df_202412_filtered

,CODEBELISTA,Composite_key,ID_OFERTA,recency,count,normalized_count,normalized_recency,score
0,4714016,[INDIVIDUAL + ADICIONAL|FIJO|200108044|0|1|1.0...,"[[1165, 1165], [1492], [734], [1436], [1489], ...","[0.0073849939514799, 0.0075187969924812, 0.007...","[2, 1, 1, 1, 1, 1, 2, 1, 1, 1]","[0.125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.125, 0.0, 0...","[0.7342404215189546, 0.8499509643674399, 0.849...","[0.3077721264556864, 0.2549852893102319, 0.254..."
1,24060276,[INDIVIDUAL + ADICIONAL|FIJO|200106279|0|1|1.0...,"[[1171, 1171], [1913]]","[0.0070693445243804, 0.0071334500152412]","[2, 1]","[0.125, 0.0]","[0.4612722865186107, 0.5167096001368758]","[0.2258816859555832, 0.1550128800410627]"
2,26079748,[INDIVIDUAL|VARIABLE|P0210136000|1|0|1.0|INDIV...,"[[133, 133, 1119, 1119], [147, 147], [1114, 11...","[0.0071666099537304, 0.0069682793505721, 0.006...","[4, 2, 2, 2, 1, 1]","[0.375, 0.125, 0.125, 0.125, 0.0, 0.0]","[0.5453857382477896, 0.3738728818643793, 0.373...","[0.4261157214743368, 0.1996618645593137, 0.199..."
3,30853652,[INDIVIDUAL|VARIABLE|P0132062000|1|0|1.0|INDIV...,"[[31, 821], [33, 994], [732], [993], [82, 728]...","[0.0073308270676691, 0.0072316207184628, 0.007...","[2, 2, 1, 1, 2, 2, 2, 1, 1, 1, 1]","[0.125, 0.125, 0.0, 0.0, 0.125, 0.125, 0.125, ...","[0.6873978424321665, 0.6016059169663279, 0.849...","[0.2937193527296499, 0.2679817750898984, 0.254..."
4,33585535,[VOLUMEN|VARIABLE|200046331|1|0|1.0|VOLUMEN|VA...,"[[1380, 1380], [113, 900], [669, 669], [137, 1...","[0.0072806169040884, 0.0071942446043165, 0.007...","[2, 2, 2, 2, 2, 1, 1, 1, 1]","[0.125, 0.125, 0.125, 0.125, 0.125, 0.0, 0.0, ...","[0.643976966187755, 0.5692837034720046, 0.5692...","[0.2806930898563264, 0.2582851110416013, 0.258..."
...,...,...,...,...,...,...,...,...
91,52480701,[SET VARIABLE|FIJO|200115274|1|0|1.0|SET VARIA...,"[[853, 853], [1406, 1406], [1380], [751], [1913]]","[0.0070329359897005, 0.0069444444444444, 0.007...","[2, 2, 1, 1, 1]","[0.125, 0.125, 0.0, 0.0, 0.0]","[0.4297868189193075, 0.3532608695652168, 0.643...","[0.2164360456757922, 0.193478260869565, 0.1931..."
92,52536197,[INDIVIDUAL|VARIABLE|P0194056000|1|0|1.0|INDIV...,"[[97, 832], [757], [834], [755], [1980], [1913...","[0.0075226244343891, 0.0073367422051632, 0.007...","[2, 1, 1, 1, 1, 1, 1]","[0.125, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.8532608695652169, 0.6925131504650954, 0.676...","[0.343478260869565, 0.2077539451395286, 0.2029..."
93,52536707,[SET FIJO|FIJO|200091328|0|0|1.0|SET FIJO|FIJO...,"[[341, 341, 341, 341, 341, 341, 341, 341, 341]...","[0.0070133587786259, 0.0069444444444444, 0.007...","[9, 3, 2, 2, 1, 1, 1, 1]","[1.0, 0.25, 0.125, 0.125, 0.0, 0.0, 0.0, 0.0]","[0.4128567872552269, 0.3532608695652168, 0.576...","[0.823857036176568, 0.280978260869565, 0.26030..."
94,52537037,[SET FIJO|FIJO|200111318|0|0|1.0|SET FIJO|FIJO...,"[[1321, 1321, 1321, 1321], [1130], [1112], [39...","[0.0075187969924812, 0.0075187969924812, 0.007...","[4, 1, 1, 1, 1]","[0.375, 0.0, 0.0, 0.0, 0.0]","[0.8499509643674399, 0.8499509643674399, 0.849...","[0.517485289310232, 0.2549852893102319, 0.2549..."


In [ ]:
# Find the repeat offers bought by a single consultora
import pandas as pd

def compare_dataframes(df1_path: str, df2_path: str) -> None:
    """
    Compare two dataframes and find matching CODEBELISTA with identical Composite_key strings.
    Args:
        df1_path: Path to first CSV file
        df2_path: Path to second CSV file
    """
    try:
        # Read CSVs
        df1 = pd.read_csv(df1_path)
        df2 = pd.read_csv(df2_path)
        
        # For each row in first DataFrame
        for idx1, row1 in df1.iterrows():
            # Find matching CODEBELISTA in second DataFrame
            matching_rows = df2[df2['CODEBELISTA'] == row1['CODEBELISTA']]
            
            if not matching_rows.empty:
                # For each matching CODEBELISTA, check if Composite_keys match exactly
                for idx2, row2 in matching_rows.iterrows():
                    if row1['Composite_key'] == row2['Composite_key']:
                        print(f"\nMatch found:")
                        print(f"CODEBELISTA: {row1['CODEBELISTA']}")
                        print(f"Composite_key: {row1['Composite_key']}")
                        
    except FileNotFoundError as e:
        print(f"Error: Could not find file - {e}")
    except pd.errors.EmptyDataError:
        print("Error: One or both CSV files are empty")
    except Exception as e:
        print(f"Error: An unexpected error occurred - {e}")

compare_dataframes("202412.csv", "202413.csv")


Match found:
CODEBELISTA: 52256704
Composite_key: INDIVIDUAL|VARIABLE|P0210133000|1|0|1.0|INDIVIDUAL|VARIABLE|P0210133000|1|0|1.0|INDIVIDUAL|VARIABLE|P0210133000|1|0|1.0|INDIVIDUAL|VARIABLE|P0210133000|1|0|1.0

Match found:
CODEBELISTA: 51758978
Composite_key: INDIVIDUAL|FIJO|200106304|1|0|1.0

Match found:
CODEBELISTA: 36266651
Composite_key: INDIVIDUAL|VARIABLE|P0132053000|1|0|1.0|INDIVIDUAL|VARIABLE|P0132053000|1|0|1.0|INDIVIDUAL|VARIABLE|P0132053000|1|0|1.0


In [62]:
# find same consultoras in both campaigns
matching_values = single_campaign_df_202412_filtered['CODEBELISTA'][single_campaign_df_202412_filtered['CODEBELISTA'].isin(single_campaign_df_202413_filtered['CODEBELISTA'])].tolist()

In [63]:
len(matching_values)

80

In [68]:
# Find the overlap of offers between previous offers and current offers
previous_offers_df = pd.read_csv("../src/data/files/Exkdabre_Exploration (1).csv")

In [69]:
def filter_invalid_offers(df):
    """
    Filter out all rows where ID_OFERTA corresponds to any row having CODCUC as 'XXXXXXXXX'
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing 'ID_OFERTA' and 'CODCUC' columns
    
    Returns:
    pandas.DataFrame: Filtered DataFrame with removed rows
    """
    # Get all ID_OFERTAs that have CODCUC as 'XXXXXXXXX'
    invalid_ids = df[df['CODCUC'] == 'XXXXXXXXX']['ID_OFERTA'].unique()
    
    # Filter out all rows with these ID_OFERTAs
    filtered_df = df[~df['ID_OFERTA'].isin(invalid_ids)]
    
    # Print some information about the filtering
    removed_count = len(df) - len(filtered_df)
    print(f"Removed {removed_count} rows")
    print(f"Found {len(invalid_ids)} unique ID_OFERTAs with CODCUC 'XXXXXXXXX'")
    
    return filtered_df

df_previous_filtered = filter_invalid_offers(previous_offers_df)

Removed 75344 rows
Found 4236 unique ID_OFERTAs with CODCUC 'XXXXXXXXX'


In [71]:
# Dropping the COMPOSITE_PRIMARY_KEY column 
df_previous_filtered = df_previous_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1)  # Remove "Column2"
print(df_previous_filtered.shape)

(51554, 22)


In [72]:
unique_occurrences = df_previous_filtered["COD_PERIODO"].value_counts()

print("Unique occurrences:")
print(unique_occurrences)

Unique occurrences:
COD_PERIODO
202418    24707
202417    18877
202416     2206
202415     2189
202412     1733
202413     1125
202414      717
Name: count, dtype: int64


In [73]:
# Creation of Composite key by combining the 6 attributes
df_previous_filtered["Composite_key"] = df_previous_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)

In [74]:
# Divide the DataFrame based on unique values in the 'COD_PERIODO' column
df_previouses = {campaign_id: df_subset for campaign_id, df_subset in df_previous_filtered.groupby('COD_PERIODO')}

In [75]:
# Assuming dfs contains the smaller DataFrames (subsets) from the previous steps
# Example of transformations on each sub_df
for i, (campaign_id, sub_df) in enumerate(df_previouses.items(), 1):
    
    # Perform the groupby and aggregation
    sub_df_concatenated = sub_df.groupby("ID_OFERTA")[["Composite_key"]].agg(
        {'Composite_key': lambda x: '|'.join(x)}
    ).reset_index()
    
    # Save the result to a CSV file with a dynamic name
    file_name = f"previous_{campaign_id}.csv"
    sub_df_concatenated.to_csv(file_name, index=False)
    
    print(f"Saved {file_name}")

Saved previous_202412.csv
Saved previous_202413.csv
Saved previous_202414.csv
Saved previous_202415.csv
Saved previous_202416.csv
Saved previous_202417.csv
Saved previous_202418.csv


In [76]:
# Calculate the percentage of offers present in the next campaign
def calculate_key_overlap_percentage(csv1_path, csv2_path):

    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)
    
    # Clean the composite keys (remove whitespace and convert to lowercase)
    # df1['Composite_key'] = df1['Composite_key'].str.strip().str.lower()
    # df2['Composite_key'] = df2['Composite_key'].str.strip().str.lower()
    
    # Get unique composite keys from both DataFrames
    keys_in_csv1 = df1['Composite_key']
    keys_in_csv2 = df2['Composite_key']
    unique_keys_in_csv1 = set(df1['Composite_key'].unique())
    unique_keys_in_csv2 = set(df2['Composite_key'].unique())
    
    # Find overlapping keys
    common_keys = unique_keys_in_csv1.intersection(unique_keys_in_csv2)
    
    # Calculate percentage
    overlap_percentage = (len(common_keys) / len(unique_keys_in_csv1)) * 100
    
    # Compile statistics
    stats = {
        'keys_in_csv1': len(keys_in_csv1),
        'keys_in_csv2': len(keys_in_csv2),
        'total_keys_csv1': len(unique_keys_in_csv1),
        'total_keys_csv2': len(unique_keys_in_csv2),
        'common_keys': len(common_keys),
        'overlap_percentage': round(overlap_percentage, 2)
    }
    
    return overlap_percentage, stats

In [78]:
csv1_path = "previous_202412.csv"
csv2_path = "previous_202413.csv"

csv1_value = os.path.splitext(os.path.basename(csv1_path))[0]
csv2_value = os.path.splitext(os.path.basename(csv2_path))[0]
percentage, stats = calculate_key_overlap_percentage(csv1_path, csv2_path)

print(f"\nResults:")
print(f"Number of offers in {csv1_value}: {stats['keys_in_csv1']}")
print(f"Total unique offers in {csv1_value}: {stats['total_keys_csv1']}")
print(f"Number of offers in {csv1_value}: {stats['keys_in_csv2']}")
print(f"Total unique offers in {csv2_value}: {stats['total_keys_csv2']}")
print(f"Number of common offers: {stats['common_keys']}")
print(f"Percentage of offers of {csv1_value} present in {csv2_value}: {stats['overlap_percentage']}%")


Results:
Number of offers in previous_202412: 554
Total unique offers in previous_202412: 503
Number of offers in previous_202412: 478
Total unique offers in previous_202413: 459
Number of common offers: 58
Percentage of offers of previous_202412 present in previous_202413: 11.53%


Filtering out duplicate offers from CAT, REV, DIG

In [11]:
df_future = pd.read_csv("../src/data/files/Exkdabre_Exploration (1).csv")

In [12]:
# For future campaigns
df_future_filtered = df_future[df_future["CODCUC"] != "XXXXXXXXX"]

print(df_future_filtered.shape)

(115085, 23)


In [13]:
df_future_filtered["Composite_key"] = df_future_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)

/var/folders/bh/74g24pss1j3_fnkxfr5pf0br0000gn/T/ipykernel_1271/2214267813.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_future_filtered["Composite_key"] = df_future_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)


In [14]:
# For future campaigns
df_future_filtered = df_future_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1)  # Remove "Column2"
print(df_future_filtered.shape)

(115085, 23)


In [15]:
unique_occurrences = df_future_filtered["COD_PERIODO"].value_counts()

print("Unique occurrences:")
print(unique_occurrences)

Unique occurrences:
COD_PERIODO
202418    35380
202417    29499
202412    10816
202415    10280
202413    10214
202416    10170
202414     8726
Name: count, dtype: int64


In [16]:
# Divide the DataFrame based on unique values in the 'COD_PERIODO' column
df_futures = {campaign_id: df_subset for campaign_id, df_subset in df_future_filtered.groupby('COD_PERIODO')}
i=1
# Dynamically create variable names for each sub_df
for campaign_id, sub_df in df_futures.items():
    # Create a unique name for each sub_df (based on category)
    variable_name = f"df_{i}"
    
    # Assign the sub_df to the unique variable name
    globals()[variable_name] = sub_df
    
    # Optionally, print the name and content to verify
    print(f"Created variable: {variable_name}")
    print(globals()[variable_name])
    print()
    i+=1


Created variable: df_1
        ID_OFERTA  COD_PERIODO  COD_CATALOGO  ID_MACROESTRATEGIA  \
69148        3438       202412          45.0                 242   
69158        3455       202412          45.0                1713   
69160        3578       202412          45.0                1758   
69165        3588       202412          45.0                1737   
69174        3556       202412          45.0                1653   
...           ...          ...           ...                 ...   
126850       3802       202412          45.0                1691   
126852       1574       202412          24.0                 643   
126868       1377       202412          24.0                 214   
126875       3247       202412          45.0                 414   
126892       3537       202412          45.0                1681   

        ID_TIPO_SUBESTRATEGIA DES_TIPO_SUBESTRATEGIA DES_TIPO_ESTRATEGIA  \
69148                       3           SET VARIABLE                 SET   
69158   

In [19]:
# Assuming dfs contains the smaller DataFrames (subsets) from the previous steps
# Example of transformations on each sub_df
all_future_dfs = []
for i, (campaign_id, sub_df) in enumerate(df_futures.items(), 1):
    
    # Perform the groupby and aggregation
    sub_df_concatenated = sub_df.groupby("ID_OFERTA")[["Composite_key"]].agg(
        {'Composite_key': lambda x: '|'.join(x)}
    ).reset_index()
    
    all_future_dfs.append(sub_df_concatenated)
    # Save the result to a CSV file with a dynamic name
    file_name = f"Future_{campaign_id}.csv"
    sub_df_concatenated.to_csv(file_name, index=False)
    
    print(f"Saved {file_name}")
final_future_df = pd.concat(all_future_dfs, ignore_index=True)

Saved Future_202412.csv
Saved Future_202413.csv
Saved Future_202414.csv
Saved Future_202415.csv
Saved Future_202416.csv
Saved Future_202417.csv
Saved Future_202418.csv


In [20]:
duplicate_offers_count = final_future_df['Composite_key'].isin(final_df['Composite_key']).sum()
print(f"Number of matching Composite_key values: {duplicate_offers_count}")

Number of matching Composite_key values: 4852
